# Descripción de las variables que se van a tratar

In [ ]:
import pandas as pd
import os
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# # Configuración de las rutas
PATH = Path().cwd()
PATH = PATH.parent if PATH.name == "notebooks" else PATH
DATA = 'data'
PROCESSED_DATA = 'processed/clean_data.csv'

print(f'La nueva ruta de los archivos ha sido asignado a: {PATH}')

In [ ]:
df = pd.read_csv(PATH / DATA / PROCESSED_DATA)

df.head()

Tendencia de los datos obtenidos en observación para cada año.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')

In [ ]:
df.head()

In [ ]:
df['canton'].unique()

In [ ]:
canton_map = {
    101: "SAN JOSE",
    102: "ESCAZU",
    103: "DESAMPARADOS",
    104: "PURISCAL",
    105: "TARRAZU",
    106: "ASERRI",
    107: "MORA",
    108: "GOICOECHEA",
    109: "SANTA ANA",
    110: "ALAJUELITA",
    111: "CORONADO",
    112: "ACOSTA",
    113: "TIBAS",
    114: "MORAVIA",
    115: "MONTES DE OCA",
    116: "TURRUBARES",
    117: "DOTA",
    118: "CURRIDABAT",
    119: "PEREZ ZELEDON",
    120: "LEON CORTES",
    201: "ALAJUELA",
    202: "SAN RAMON",
    203: "GRECIA",
    204: "SAN MATEO",
    205: "ATENAS",
    206: "NARANJO",
    207: "PALMARES",
    208: "POAS",
    209: "OROTINA",
    210: "SAN CARLOS",
    211: "ALFARO RUIZ",
    212: "VALVERDE VEGA",
    213: "UPALA",
    214: "LOS CHILES",
    215: "GUATUSO",
    301: "CARTAGO",
    302: "PARAISO",
    303: "LA UNION",
    304: "JIMENEZ",
    305: "TURRIALBA",
    306: "ALVARADO",
    307: "OREAMUNO",
    308: "EL GUARCO",
    401: "HEREDIA",
    402: "BARVA",
    403: "SANTO DOMINGO",
    404: "SANTA BARBARA",
    405: "SAN RAFAEL",
    406: "SAN ISIDRO",
    407: "BELEN",
    408: "FLORES",
    409: "SAN PABLO",
    410: "SARAPIQUI",
    501: "LIBERIA",
    502: "NICOYA",
    503: "SANTA CRUZ",
    504: "BAGACES",
    505: "CARRILLO",
    506: "CAÑAS",
    507: "ABANGARES",
    508: "TILARAN",
    509: "NANDAYURE",
    510: "LA CRUZ",
    511: "HOJANCHA",
    601: "PUNTARENAS",
    602: "ESPARZA",
    603: "BUENOS AIRES",
    604: "MONTES DE ORO",
    605: "OSA",
    606: "AGUIRRE",
    607: "GOLFITO",
    608: "COTO BRUS",
    609: "PARRITA",
    610: "CORREDORES",
    611: "GARABITO",
    701: "LIMON",
    702: "POCOCI",
    703: "SIQUIRRES",
    704: "TALAMANCA",
    705: "MATINA",
    706: "GUACIMO"
}

In [ ]:
df['canton'] = df['canton'].map(canton_map)

En la siguiente figura se observa que no hay tendencia crecientes o estacionalidad. En cambio, se observa cantones tienen cierto cíclo, pero este no ha sido constante dentro de cada año o por año.

In [ ]:
totales = df.groupby('canton')['casos'].sum().sort_values(ascending=False)

top5 = totales.head(5).index

plt.figure(figsize=(12,6))

for canton, subset in df.groupby('canton'):
    if canton in top5:
        plt.plot(subset['date'], subset['casos'], label=f"{canton}")
    else:
        plt.plot(subset['date'], subset['casos'], color="lightgray", alpha=0.7)

plt.xlabel("Fecha")
plt.ylabel("Casos")
plt.title("Serie temporal de casos (Top 5 cantones destacados)")
plt.xticks(rotation=45)
plt.legend(title="Top 5 Cantones")
plt.grid(False)
plt.show()

## Selección de variables que tienen mayor importancia

En este caso se van a tomar en cuenta aquellas variables que tienen mayor importancia para el número de casos en Costa Rica.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor
from category_encoders import TargetEncoder

In [ ]:
targetEncoder = TargetEncoder(smoothing=10.0, min_samples_leaf=50) # min_samples_leaf: En caso que hayan muy pocos casos en un cantón

In [ ]:
# Preparando los conjuntos de datos con su nueva codificación

df_exploracion = df.copy()

df_exploracion['canton_code'] = targetEncoder.fit_transform(
    df_exploracion['canton'], 
    df_exploracion['casos']
)

In [ ]:
X_explor = df_exploracion.drop(columns=['canton', 'casos', 'date'])
y_explor = df_exploracion['casos']

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=1234, n_jobs=-1)
rf.fit(X_explor, y_explor)

In [ ]:
result = permutation_importance(rf, X_explor, y_explor, n_repeats=5, random_state=1234, n_jobs=-1)

In [ ]:
df_importancias = pd.DataFrame({
    'variable': X_explor.columns,
    'importancia_media': result.importances_mean,
    'importancia_std': result.importances_std
})

df_importancias = df_importancias.sort_values(by='importancia_media', ascending=False).reset_index(drop=True)

df_top = df_importancias[df_importancias['importancia_media'] > 0].head(10)

En la siguiente figura se observa que los ``lags`` son los que se están llevando la mayor importancia. Al permutar casos_lag_1, el rendimiento del modelo se destruye por completo (cae más de $1.4$), mientras que permutar la lluvia (rr) apenas le hace comparación. Eso no significa que el clima o el cantón no importen, significa que su información ya está indirectamente contenida dentro del lag. El lag es el resultado histórico de la combinación de clima + geografía de la semana/mes pasada.

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 8))

plt.barh(
    y=df_top['variable'][::-1],
    width=df_top['importancia_media'][::-1],
    xerr=df_top['importancia_std'][::-1],
    color='skyblue',
    edgecolor='gray',
    alpha=0.8
)

plt.xlabel('Descenso en el rendimiento del modelo (Permutation Importance)')
plt.ylabel('Variables')
plt.title('Top 20 Variables Más Influyentes en la Predicción del Dengue', fontsize=14, pad=15)
plt.tight_layout()
plt.grid(False)
plt.show()

+ Eliminación de `lags`.

En la figura anterior vimos que la información contenidad ya nos mostraba que los lags se llevan la
mayor importancia. Ante esto, se decide quitar todos los lags, esto como experimento, y así observar cómo se comporta el fenomeno.

In [ ]:
variables_sin_lags = [col for col in df.columns if '_lag_' not in col]

X_explor = df[variables_sin_lags].drop(columns=['casos', 'canton', 'date']) 
y_explor = df['casos']

In [ ]:
rf2 = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf2.fit(X_explor, y_explor)

In [ ]:
result = permutation_importance(rf2, X_explor, y_explor, n_repeats=5, random_state=1234, n_jobs=-1)

In [ ]:
df_importancias = pd.DataFrame({
    'variable': X_explor.columns,
    'importancia_media': result.importances_mean,
    'importancia_std': result.importances_std
})

df_importancias = df_importancias.sort_values(by='importancia_media', ascending=False).reset_index(drop=True)

df_top = df_importancias[df_importancias['importancia_media'] > 0].head(20)

En el siguiente figura se observa que las variables como lluvia, fénomeno del niño, la estacionalidad del año y las temperaturas, son aquellas las
que muestran una fuerza comprender el fénomeno de estudio. sin embargo, se debe de realizar la alusión que las el cantón no ha salido como relevante en este análisis previo.

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 8))

plt.barh(
    y=df_top['variable'][::-1],
    width=df_top['importancia_media'][::-1],
    xerr=df_top['importancia_std'][::-1],
    color='skyblue',
    edgecolor='gray',
    alpha=0.8
)

plt.xlabel('Descenso en el rendimiento del modelo (Permutation Importance) - Sin Lags')
plt.ylabel('Variables')
plt.title('Top 20 Variables Más Influyentes en la Predicción del Dengue', fontsize=14, pad=15)
plt.tight_layout()
plt.grid(False)
plt.show()

## Visualización de las 5 variables más importantes

rr, nino34ssta, week, tmax_mean, tmin_mean

## El Ciclo Anual (Estacionalidad): Gráfico de Doble Eje Y (week vs. rr)

In [ ]:
df_semana = df.groupby('week').agg({
    'casos': 'mean',
    'rr': 'mean'
}).reset_index()

In [ ]:
sns.set_theme(style="white")
fig, ax1 = plt.subplots(figsize=(14, 6))

# Eje izquierdo (Azul) - Casos de Dengue
color = '#2b7bba'
ax1.set_xlabel('Semana Epidemiológica', fontsize=12, labelpad=10)
ax1.set_ylabel('Promedio de Casos de Dengue', color=color, fontsize=12)
sns.lineplot(data=df_semana, x='week', y='casos', ax=ax1, color=color, linewidth=2.5, marker='o')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(range(1, 53, 2))

# Eje derecho (Verde/Gris) - Lluvia (rr)
ax2 = ax1.twinx()  
color = '#41ab5d'
ax2.set_ylabel('Promedio de Precipitación (rr)', color=color, fontsize=12)
sns.lineplot(data=df_semana, x='week', y='rr', ax=ax2, color=color, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Análisis de Estacionalidad: Comportamiento del Dengue vs. Precipitación por Semana', fontsize=14, pad=15)
fig.tight_layout()
plt.show()

## 2. El Impacto Macroclimático: Gráfico de Dispersión con `nino34ssta`

In [ ]:
df_nino = df.groupby(['year', 'week']).agg({
    'casos': 'sum',
    'nino34ssta': 'mean'
}).reset_index()

In [ ]:
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

sns.regplot(
    data=df_nino, 
    x='nino34ssta', 
    y='casos',
    scatter_kws={'alpha':0.5, 'color': '#d95f02'},
    line_kws={'color': '#7570b3', 'linewidth': 2},
    lowess=True # Ajuste suavizado no lineal para capturar curvas si existen
)

plt.xlabel('Anomalía de la TSM - Índice El Niño 3.4 (nino34ssta)', fontsize=12)
plt.ylabel('Total de Casos de Dengue', fontsize=12)
plt.title('Relación entre las Anomalías de El Niño (ENOS) y el Volumen de Casos de Dengue', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

## 3. El Umbral Térmico: Gráficos de Densidad para ``tmax_mean`` y ``tmin_mean``

In [ ]:
import numpy as np

In [ ]:
umbral_alto = df['casos'].quantile(0.75)

df_termico = df.copy()
df_termico['Estado del Brote'] = np.where(df_termico['casos'] >= umbral_alto, 'Brote Alto (>= P75)', 'Brote Bajo/Normal')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Gráfico para Temperatura Máxima
sns.kdeplot(data=df_termico, x='tmax_mean', hue='Estado del Brote', fill=True, common_norm=False, palette='coolwarm', alpha=0.4, ax=axes[0])
axes[0].set_title('Distribución de Temperatura Máxima Promedio', fontsize=12)
axes[0].set_xlabel('tmax_mean (°C)')
axes[0].set_ylabel('Densidad')

# Gráfico para Temperatura Mínima
sns.kdeplot(data=df_termico, x='tmin_mean', hue='Estado del Brote', fill=True, common_norm=False, palette='coolwarm', alpha=0.4, ax=axes[1])
axes[1].set_title('Distribución de Temperatura Mínima Promedio', fontsize=12)
axes[1].set_xlabel('tmin_mean (°C)')
axes[1].set_ylabel('')

plt.suptitle('Umbrales Térmicos: Comparativa de Temperaturas según la Intensidad del Brote de Dengue', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()